<a href="https://colab.research.google.com/github/Lyv-ux/DI_Bootcamp/blob/main/W7D3_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# DAILY CHALLENGE - BUILD A RAG SYSTEM WITH LANGCHAIN
# Tout dans une seule cellule avec commentaires
# ============================================================

# ============================================================
# ETAPE 1 - Installer les bibliothèques nécessaires
# (Décommente ces lignes la première fois seulement)
# ============================================================

# !pip install -q langchain
# !pip install -q torch
# !pip install -q transformers
# !pip install -q sentence-transformers
# !pip install -q datasets
# !pip install -q faiss-cpu
# !pip install -U langchain-community

# ============================================================
# ETAPE 2 - Importer les bibliothèques
# ============================================================

from langchain_community.document_loaders import HuggingFaceDatasetLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    pipeline
)

from langchain_community.llms import HuggingFacePipeline
from langchain.chains import RetrievalQA

# ============================================================
# ETAPE 3 - Charger le dataset Dolly de Hugging Face
# ============================================================

dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(
    dataset_name=dataset_name,
    page_content_column=page_content_column
)

data = loader.load()

print("=" * 60)
print("FIRST 2 DOCUMENTS")
print("=" * 60)
print(data[:2])

# ============================================================
# ETAPE 4 - Découper les documents en chunks
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

docs = text_splitter.split_documents(data)

print("\nNumber of chunks:", len(docs))
print("\nFirst chunk:\n")
print(docs[0])

# ============================================================
# ETAPE 5 - Créer les embeddings avec MiniLM
# ============================================================

modelPath = "sentence-transformers/all-MiniLM-L6-v2"

model_kwargs = {
    "device": "cpu"
}

encode_kwargs = {
    "normalize_embeddings": False
}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

# ============================================================
# ETAPE 6 - Tester la création d'un embedding
# ============================================================

text = "This is a test document."

query_result = embeddings.embed_query(text)

print("\nEmbedding sample (first 3 values):")
print(query_result[:3])

# ============================================================
# ETAPE 7 - Créer la base vectorielle FAISS
# ============================================================

print("\nCreating FAISS vector database...")

db = FAISS.from_documents(
    docs,
    embeddings
)

print("FAISS database created successfully.")

# ============================================================
# ETAPE 8 - Charger le modèle Question Answering
# ============================================================

model_name = "Intel/dynamic_tinybert"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    padding=True,
    truncation=True,
    max_length=512
)

model = AutoModelForQuestionAnswering.from_pretrained(
    model_name
)

# ============================================================
# ETAPE 9 - Créer le pipeline Hugging Face
# ============================================================

Youtubeer = pipeline(
    "question-answering",
    model=model_name,
    tokenizer=tokenizer,
    return_tensors="pt"
)

# ============================================================
# ETAPE 10 - Créer le wrapper LangChain
# ============================================================

llm = HuggingFacePipeline(
    pipeline=Youtubeer,
    model_kwargs={
        "temperature": 0.7,
        "max_length": 512
    }
)

# ============================================================
# ETAPE 11 - Créer le retriever
# ============================================================

retriever = db.as_retriever(
    search_kwargs={"k": 4}
)

# ============================================================
# ETAPE 12 - Construire la chaîne RAG
# ============================================================

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=retriever,
    return_source_documents=False
)

# ============================================================
# ETAPE 13 - Tester le système RAG
# ============================================================

question = "What is cheesemaking?"

print("\n" + "=" * 60)
print("QUESTION")
print("=" * 60)
print(question)

result = qa.run(
    {"query": question}
)

print("\n" + "=" * 60)
print("ANSWER")
print("=" * 60)
print(result)